# Home Credit Default Risk — data download & EDA

Competition: <https://www.kaggle.com/competitions/home-credit-default-risk/data>

**Goal:** predict `TARGET` (1 = client had payment difficulties) for each loan application.

**Data layout:** one main table (`application_train/test`, one row per loan application, ~120 features) plus 6 history tables
joined on `SK_ID_CURR` (and `SK_ID_BUREAU` / `SK_ID_PREV`):

| table | grain |
|---|---|
| `bureau` | previous credits at other institutions (Credit Bureau) |
| `bureau_balance` | monthly status of each bureau credit |
| `previous_application` | previous Home Credit applications |
| `POS_CASH_balance` | monthly snapshots of previous POS / cash loans |
| `credit_card_balance` | monthly snapshots of previous credit cards |
| `installments_payments` | payment history of previous Home Credit loans |

Because there are hundreds of columns, the EDA **selects the main features** (LightGBM gain + univariate AUC) and focuses on them,
then summarises each history table with a few aggregated, interpretable per-client signals.

Kaggle auth: the notebook reads `kaggle_api_token=...` from `./.env` (or uses `~/.kaggle/kaggle.json`).
You must have accepted the competition rules on Kaggle.

## 0. Setup

In [ ]:
import os, time, zipfile, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.4f}'.format)

DATA_DIR = Path('./data')
COMP = 'home-credit-default-risk'
MAX_BYTES = 5 * 1024**3          # hard limit for ./data

# plotting style: one fixed color per class, recessive axes/grid
C_REPAID, C_DEFAULT = '#2a78d6', '#eb6834'   # TARGET=0 / TARGET=1
C_BAR = '#2a78d6'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
DIVERGING = LinearSegmentedColormap.from_list('div', ['#2a78d6', '#f0efec', '#eb6834'])
plt.rcParams.update({
    'figure.dpi': 100, 'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': INK2, 'axes.labelcolor': INK2, 'axes.titlecolor': INK, 'axes.titlesize': 11,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': GRID, 'grid.linewidth': 0.8, 'xtick.color': INK2, 'ytick.color': INK2,
    'lines.linewidth': 2, 'legend.frameon': False, 'font.size': 9,
})

## 1. Download data to `./data` (≤ 5 GB)

Skipped if the CSVs are already there. Before downloading it checks the remote total size against the 5 GB budget
(and again on the zip's uncompressed size), then extracts and deletes the zip.

In [ ]:
EXPECTED = ['application_train.csv', 'application_test.csv', 'bureau.csv', 'bureau_balance.csv',
            'previous_application.csv', 'POS_CASH_balance.csv', 'credit_card_balance.csv',
            'installments_payments.csv', 'HomeCredit_columns_description.csv', 'sample_submission.csv']

def load_env_token(path='.env'):
    # Put kaggle_api_token from .env into KAGGLE_API_TOKEN (read by kaggle>=1.8).
    if os.environ.get('KAGGLE_API_TOKEN') or not Path(path).exists():
        return
    for line in Path(path).read_text().splitlines():
        k, _, v = line.partition('=')
        if k.strip().lower() == 'kaggle_api_token':
            os.environ['KAGGLE_API_TOKEN'] = v.strip().strip('"').strip("'")

def retry(fn, tries=4, wait=5):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:          # network hiccups (seen: intermittent SSL errors)
            status = getattr(getattr(e, 'response', None), 'status_code', None)
            if status == 403:
                raise RuntimeError('Kaggle returned 403: accept the competition rules first at '
                                   f'https://www.kaggle.com/competitions/{COMP}/rules (Join / I Understand and Accept)') from e
            if i == tries - 1 or status in (400, 401, 404):
                raise
            print(f'  retry {i + 1}/{tries - 1} after error: {type(e).__name__}: {str(e)[:120]}')
            time.sleep(wait)

def dir_size(p):
    return sum(f.stat().st_size for f in Path(p).rglob('*') if f.is_file())

DATA_DIR.mkdir(exist_ok=True)
if all((DATA_DIR / f).exists() for f in EXPECTED):
    print('Data already present - skipping download.')
else:
    load_env_token()
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    retry(api.authenticate)
    listing = retry(lambda: api.competition_list_files(COMP))
    files = getattr(listing, 'files', listing)
    remote_total = sum(f.total_bytes for f in files)
    existing = dir_size(DATA_DIR)
    print(f'Remote files: {len(files)}, uncompressed total {remote_total / 1e9:.2f} GB; already in ./data: {existing / 1e9:.2f} GB')
    if existing + remote_total > MAX_BYTES:
        raise RuntimeError('Download would exceed the 5 GB limit for ./data')
    if shutil.disk_usage(DATA_DIR).free < remote_total * 1.5:
        raise RuntimeError('Not enough free disk space')

    retry(lambda: api.competition_download_files(COMP, path=str(DATA_DIR), quiet=False))
    zip_path = DATA_DIR / f'{COMP}.zip'
    with zipfile.ZipFile(zip_path) as z:
        uncompressed = sum(i.file_size for i in z.infolist())
        if existing + zip_path.stat().st_size + uncompressed > MAX_BYTES:
            raise RuntimeError('Extraction would exceed the 5 GB limit for ./data')
        z.extractall(DATA_DIR)
    zip_path.unlink()

sizes = pd.Series({f.name: f.stat().st_size / 1e6 for f in sorted(DATA_DIR.glob('*.csv'))}, name='MB')
print(sizes.round(1).to_string())
print(f'\nTotal in ./data: {dir_size(DATA_DIR) / 1e9:.2f} GB (limit 5 GB)')

## 2. Table overview

In [ ]:
def n_rows(path):
    with open(path, 'rb') as f:
        return sum(buf.count(b'\n') for buf in iter(lambda: f.read(1 << 24), b'')) - 1

overview = pd.DataFrame([
    {'table': p.stem, 'rows': n_rows(p), 'cols': len(pd.read_csv(p, nrows=0, encoding='latin-1').columns),
     'size_MB': p.stat().st_size / 1e6}
    for p in sorted(DATA_DIR.glob('*.csv')) if p.name != 'HomeCredit_columns_description.csv'
]).set_index('table')

# column descriptions (file is latin-1 encoded)
desc = pd.read_csv(DATA_DIR / 'HomeCredit_columns_description.csv', encoding='latin-1', index_col=0)
DESC = desc.drop_duplicates('Row').set_index('Row')['Description'].to_dict()
overview

## 3. Main table: `application_train`

In [ ]:
train = pd.read_csv(DATA_DIR / 'application_train.csv')
test = pd.read_csv(DATA_DIR / 'application_test.csv')
print('train', train.shape, f'{train.memory_usage(deep=True).sum() / 1e6:.0f} MB')
print('test ', test.shape)
print('\ndtypes:', train.dtypes.value_counts().to_dict())
print('duplicate SK_ID_CURR:', train.SK_ID_CURR.duplicated().sum())
train.head()

### 3.1 Target

In [ ]:
rate = train.TARGET.mean()
counts = train.TARGET.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 2.2))
ax.barh(['Repaid (0)', 'Default (1)'], counts.values, color=[C_REPAID, C_DEFAULT], height=0.55)
for i, v in enumerate(counts.values):
    ax.text(v, i, f'  {v:,}  ({v / counts.sum():.1%})', va='center', color=INK)
ax.set_xlim(0, counts.max() * 1.35); ax.grid(axis='y', visible=False)
ax.set_title(f'TARGET is imbalanced: {rate:.2%} defaults')
plt.show()

### 3.2 Cleaning & derived columns
* `DAYS_*` are negative days relative to the application → convert age/employment to positive years.
* `DAYS_EMPLOYED = 365243` (~1000 years) is a placeholder (mostly pensioners) → set NaN + flag.
* A few standard ratio features (credit/income, annuity/income, credit term, goods/credit, mean of EXT_SOURCE).

In [ ]:
anom = train.DAYS_EMPLOYED == 365243
print(f'DAYS_EMPLOYED anomaly rows: {anom.sum():,} ({anom.mean():.1%}); '
      f'default rate anomalous={train.TARGET[anom].mean():.2%} vs others={train.TARGET[~anom].mean():.2%}')
print('Income types of the anomalous rows:', train.loc[anom, 'NAME_INCOME_TYPE'].value_counts().to_dict())

def prep(df):
    df = df.copy()
    df['DAYS_EMPLOYED_ANOM'] = (df.DAYS_EMPLOYED == 365243).astype(int)
    df['DAYS_EMPLOYED'] = df.DAYS_EMPLOYED.replace(365243, np.nan)
    df['AGE_YEARS'] = -df.DAYS_BIRTH / 365.25
    df['YEARS_EMPLOYED'] = -df.DAYS_EMPLOYED / 365.25
    df['CREDIT_INCOME_RATIO'] = df.AMT_CREDIT / df.AMT_INCOME_TOTAL
    df['ANNUITY_INCOME_RATIO'] = df.AMT_ANNUITY / df.AMT_INCOME_TOTAL
    df['CREDIT_TERM'] = df.AMT_ANNUITY / df.AMT_CREDIT          # ~ 1 / number of payments
    df['GOODS_CREDIT_RATIO'] = df.AMT_GOODS_PRICE / df.AMT_CREDIT
    df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
    return df.drop(columns=['DAYS_BIRTH'])

train, test = prep(train), prep(test)
ENGINEERED = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM', 'GOODS_CREDIT_RATIO', 'EXT_SOURCE_MEAN']
DESC.update({'AGE_YEARS': 'Client age in years', 'YEARS_EMPLOYED': 'Years at current job (NaN for 365243 placeholder)',
             'DAYS_EMPLOYED_ANOM': 'DAYS_EMPLOYED was the 365243 placeholder',
             'CREDIT_INCOME_RATIO': 'AMT_CREDIT / AMT_INCOME_TOTAL', 'ANNUITY_INCOME_RATIO': 'AMT_ANNUITY / AMT_INCOME_TOTAL',
             'CREDIT_TERM': 'AMT_ANNUITY / AMT_CREDIT', 'GOODS_CREDIT_RATIO': 'AMT_GOODS_PRICE / AMT_CREDIT',
             'EXT_SOURCE_MEAN': 'Mean of EXT_SOURCE_1..3'})

### 3.3 Missing values

In [ ]:
miss = train.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print(f'{len(miss)} of {train.shape[1]} columns have missing values; {(miss > 0.5).sum()} are >50% missing '
      '(mostly the *_AVG/_MODE/_MEDI building-description block).')

top = miss.head(40)[::-1]
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(top.index, top.values * 100, color=C_BAR, height=0.6)
ax.set_xlabel('% missing'); ax.set_title('Top 40 columns by missing share'); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

## 4. Choosing the main features

Two complementary rankings on the raw application columns: a quick LightGBM model (gain importance, captures non-linearity & interactions) and univariate ROC-AUC (how well a single column separates the classes on its own).

In [ ]:
y = train.TARGET
X = train.drop(columns=['SK_ID_CURR', 'TARGET'] + ENGINEERED)
CAT_COLS = X.select_dtypes('object').columns.tolist()
for c in CAT_COLS:
    X[c] = X[c].astype('category')

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=31, colsample_bytree=0.5,
                           subsample=0.8, subsample_freq=1, reg_lambda=1.0, verbose=-1, random_state=42)
model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='auc',
          callbacks=[lgb.early_stopping(100, verbose=False)])
va_auc = roc_auc_score(y_va, model.predict_proba(X_va)[:, 1])
print(f'Baseline LightGBM on application table only: validation AUC = {va_auc:.4f} ({model.best_iteration_} trees)')

imp = pd.Series(model.booster_.feature_importance('gain'), index=X.columns)
imp = (imp / imp.sum()).sort_values(ascending=False)

In [ ]:
def uni_auc(x, y):
    # direction-free univariate AUC; NaNs filled with median (categoricals: target-rate encoding)
    if x.dtype == object or str(x.dtype) == 'category':
        x = x.astype(str).map(y.groupby(x.astype(str)).mean())
    x = x.fillna(x.median())
    a = roc_auc_score(y, x)
    return max(a, 1 - a), ('+' if a >= 0.5 else '-')

rank = pd.DataFrame({'gain_share': imp})
rank = rank.join(pd.DataFrame([uni_auc(train[c], y) for c in rank.index], index=rank.index,
                              columns=['uni_auc', 'direction']))
rank['missing'] = train[rank.index].isna().mean()
rank['description'] = [DESC.get(c, '') for c in rank.index]

TOP_N = 25
print('direction "+" = higher value -> more defaults')
rank.head(TOP_N).style.format({'gain_share': '{:.3f}', 'uni_auc': '{:.3f}', 'missing': '{:.1%}'}) \
    .bar(subset=['gain_share'], color='#9ec5f4')

In [ ]:
top = imp.head(TOP_N)[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top.index, top.values, color=C_BAR, height=0.6)
ax.set_xlabel('share of total gain'); ax.grid(axis='y', visible=False)
ax.set_title(f'LightGBM gain importance - top {TOP_N} (val AUC {va_auc:.3f})')
plt.tight_layout(); plt.show()

cum = imp.cumsum()
print(f'Top 10 features carry {cum.iloc[9]:.0%} of total gain, top 25 carry {cum.iloc[24]:.0%}, '
      f'{(cum < 0.95).sum() + 1} of {len(imp)} features reach 95%.')

In [ ]:
# Main features used for the rest of the EDA: top by gain, split into numeric (many distinct values) and categorical/low-cardinality.
def is_categorical(c):
    return c in CAT_COLS or train[c].nunique() <= 15

ranked = imp.index.tolist()
MAIN_NUM = [c for c in ranked if not is_categorical(c)][:12]
MAIN_CAT = [c for c in ranked if is_categorical(c)][:9]
print('MAIN_NUM:', MAIN_NUM)
print('MAIN_CAT:', MAIN_CAT)

## 5. Main numeric features vs TARGET

Left: distribution by class (clipped to the 1st-99th percentile). Right: default rate per decile — the most direct read of the risk relationship.

In [ ]:
def rate_by_bin(x, y, q=10):
    b = pd.qcut(x, q, duplicates='drop')
    g = y.groupby(b, observed=True).agg(['mean', 'size'])
    g.index = [f'{iv.left:.3g}-{iv.right:.3g}' for iv in g.index]
    return g

def plot_numeric(col, df=None, target=None):
    df = train if df is None else df
    x = df[col]; tg = df.TARGET if target is None else target
    lo, hi = x.quantile([0.01, 0.99])
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3), gridspec_kw={'width_ratios': [1, 1.3]})
    for t, c, lab in [(0, C_REPAID, 'Repaid'), (1, C_DEFAULT, 'Default')]:
        sns.kdeplot(x[(tg == t)].clip(lo, hi).dropna(), ax=a1, color=c, label=lab, linewidth=2, common_norm=False)
    a1.set_title(f'{col} - distribution'); a1.set_xlabel(''); a1.legend()
    g = rate_by_bin(x, tg)
    a2.plot(range(len(g)), g['mean'] * 100, color=C_BAR, marker='o', markersize=5)
    a2.axhline(tg.mean() * 100, color=INK2, linestyle='--', linewidth=1)
    a2.text(len(g) - 0.5, tg.mean() * 100, ' overall', color=INK2, va='bottom', ha='right')
    a2.set_xticks(range(len(g))); a2.set_xticklabels(g.index, rotation=35, ha='right', fontsize=7)
    a2.set_ylabel('default rate %'); a2.set_ylim(bottom=0)
    na = x.isna().mean()
    na_txt = f'  |  missing {na:.0%} (default rate when missing: {tg[x.isna()].mean():.1%})' if na > 0 else ''
    a2.set_title(f'default rate by decile{na_txt}')
    fig.suptitle(DESC.get(col, ''), x=0.01, ha='left', fontsize=9, color=INK2)
    plt.tight_layout(); plt.show()

for c in MAIN_NUM:
    plot_numeric(c)

### 5.1 Outliers / sanity checks on amounts

In [ ]:
amt = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'AGE_YEARS', 'YEARS_EMPLOYED', 'OWN_CAR_AGE', 'CNT_CHILDREN']
train[amt].describe(percentiles=[0.01, 0.5, 0.99]).T

## 6. Main categorical / low-cardinality features vs TARGET

Bars = default rate per category (top 12 by count); label shows the number of applications. Dashed line = overall rate.

In [ ]:
def plot_categorical(cols, df=None, ncols=3, top=12):
    df = train if df is None else df
    rows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(rows, ncols, figsize=(5.2 * ncols, 3.4 * rows))
    for ax, col in zip(np.ravel(axes), cols):
        s = df[col].astype(str).replace('nan', 'missing')
        g = df.TARGET.groupby(s).agg(['mean', 'size']).sort_values('size', ascending=False).head(top)
        g = g.sort_values('mean')
        ax.barh([str(i)[:28] for i in g.index], g['mean'] * 100, color=C_BAR, height=0.6)
        for i, (m, n) in enumerate(zip(g['mean'], g['size'])):
            ax.text(m * 100, i, f' {n:,}', va='center', fontsize=7, color=INK2)
        ax.axvline(df.TARGET.mean() * 100, color=INK2, linestyle='--', linewidth=1)
        ax.set_title(col); ax.set_xlabel('default rate %'); ax.grid(axis='y', visible=False)
        ax.set_xlim(0, g['mean'].max() * 100 * 1.3)
        ax.tick_params(axis='y', labelsize=7)
    for ax in np.ravel(axes)[len(cols):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

extra = [c for c in ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE',
                     'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_FAMILY_STATUS', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
                     'REGION_RATING_CLIENT_W_CITY', 'DAYS_EMPLOYED_ANOM'] if c not in MAIN_CAT]
plot_categorical(MAIN_CAT + extra[:max(0, 12 - len(MAIN_CAT))])

## 7. Engineered ratio features

In [ ]:
eng_auc = pd.DataFrame({c: uni_auc(train[c], y) for c in ENGINEERED + ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY',
                                                                        'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']},
                       index=['uni_auc', 'direction']).T.sort_values('uni_auc', ascending=False)
print(eng_auc.to_string())
for c in ['EXT_SOURCE_MEAN', 'CREDIT_TERM', 'GOODS_CREDIT_RATIO', 'CREDIT_INCOME_RATIO']:
    plot_numeric(c)

## 8. Correlations among main numeric features

In [ ]:
cols = list(dict.fromkeys(['TARGET'] + MAIN_NUM + ['EXT_SOURCE_MEAN', 'CREDIT_TERM', 'CREDIT_INCOME_RATIO']))
corr = train[cols].corr(method='spearman')
fig, ax = plt.subplots(figsize=(10, 8.5))
sns.heatmap(corr, cmap=DIVERGING, center=0, vmin=-1, vmax=1, annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=1, linecolor='#fcfcfb', cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_title('Spearman correlation - main numeric features')
plt.tight_layout(); plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
print('Most redundant pairs (|rho| > 0.8):')
print(pairs[pairs.abs() > 0.8].sort_values(key=abs, ascending=False).round(3).to_string())

## 9. Train vs test drift on main features

KS statistic between train and test distributions (0 = identical). Large values would mean the test set is shifted.

In [ ]:
drift = pd.DataFrame({c: ks_2samp(train[c].dropna(), test[c].dropna()) for c in MAIN_NUM},
                     index=['ks_stat', 'p_value']).T.sort_values('ks_stat', ascending=False)
drift['train_missing'] = train[MAIN_NUM].isna().mean(); drift['test_missing'] = test[MAIN_NUM].isna().mean()
print(drift.round(4).to_string())
print('\nNAME_CONTRACT_TYPE share - train:', train.NAME_CONTRACT_TYPE.value_counts(normalize=True).round(3).to_dict(),
      '| test:', test.NAME_CONTRACT_TYPE.value_counts(normalize=True).round(3).to_dict())

worst = drift.index[:3]
fig, axes = plt.subplots(1, 3, figsize=(13, 2.8))
for ax, c in zip(axes, worst):
    lo, hi = train[c].quantile([0.01, 0.99])
    sns.kdeplot(train[c].clip(lo, hi).dropna(), ax=ax, color=C_REPAID, label='train')
    sns.kdeplot(test[c].clip(lo, hi).dropna(), ax=ax, color='#1baf7a', label='test')
    ax.set_title(f'{c} (KS {drift.loc[c, "ks_stat"]:.3f})'); ax.set_xlabel(''); ax.legend()
plt.tight_layout(); plt.show()

## 10. History tables

Each table is summarised into a few interpretable per-client (`SK_ID_CURR`) aggregates, joined to `train`,
and scored by univariate AUC. Only the columns needed are loaded to keep memory low.

In [ ]:
hist = train[['SK_ID_CURR', 'TARGET']].copy()
def add(feat_df):
    global hist
    hist = hist.merge(feat_df, left_on='SK_ID_CURR', right_index=True, how='left')

def has_history(col):
    print(f'  train clients with records: {hist[col].notna().mean():.1%}')

### 10.1 `bureau` + `bureau_balance` — credits at other institutions

In [ ]:
bureau = pd.read_csv(DATA_DIR / 'bureau.csv')
print(bureau.shape); display(bureau.head(3))
print(bureau.CREDIT_ACTIVE.value_counts().to_dict())
print(bureau.CREDIT_TYPE.value_counts().head(6).to_dict())

b = bureau.assign(
    active=bureau.CREDIT_ACTIVE.eq('Active'),
    overdue=bureau.CREDIT_DAY_OVERDUE.gt(0),
    debt_active=bureau.AMT_CREDIT_SUM_DEBT.where(bureau.CREDIT_ACTIVE.eq('Active')),
)
bagg = b.groupby('SK_ID_CURR').agg(
    BUR_N_CREDITS=('SK_ID_BUREAU', 'size'),
    BUR_N_ACTIVE=('active', 'sum'),
    BUR_ACTIVE_SHARE=('active', 'mean'),
    BUR_ANY_OVERDUE=('overdue', 'max'),
    BUR_DEBT_ACTIVE_SUM=('debt_active', 'sum'),
    BUR_DAYS_CREDIT_MAX=('DAYS_CREDIT', 'max'),     # most recent bureau credit (days before application)
    BUR_DAYS_CREDIT_MIN=('DAYS_CREDIT', 'min'),     # oldest
)
bagg['BUR_ANY_OVERDUE'] = bagg.BUR_ANY_OVERDUE.astype(int)

# bureau_balance: worst monthly DPD status per bureau credit (0 = on time, 1..5 = 1-30 .. 120+ days late)
bb = pd.read_csv(DATA_DIR / 'bureau_balance.csv', usecols=['SK_ID_BUREAU', 'STATUS'])
print('\nbureau_balance', bb.shape, 'STATUS share:', bb.STATUS.value_counts(normalize=True).round(3).to_dict())
bb['dpd'] = pd.to_numeric(bb.STATUS, errors='coerce')          # 'C' (closed) / 'X' (unknown) -> NaN
worst = bb.groupby('SK_ID_BUREAU').dpd.max()
del bb
bagg['BUR_WORST_DPD_STATUS'] = bureau[['SK_ID_BUREAU', 'SK_ID_CURR']].assign(w=bureau.SK_ID_BUREAU.map(worst)) \
    .groupby('SK_ID_CURR').w.max()
add(bagg); has_history('BUR_N_CREDITS')
del b, bureau

### 10.2 `previous_application` — earlier Home Credit applications

In [ ]:
prev = pd.read_csv(DATA_DIR / 'previous_application.csv',
                   usecols=['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'AMT_APPLICATION', 'AMT_CREDIT',
                            'DAYS_DECISION', 'NAME_CONTRACT_TYPE', 'CNT_PAYMENT'])
print(prev.shape, prev.NAME_CONTRACT_STATUS.value_counts().to_dict())
p = prev.assign(refused=prev.NAME_CONTRACT_STATUS.eq('Refused'),
                approved=prev.NAME_CONTRACT_STATUS.eq('Approved'),
                got_less=(prev.AMT_CREDIT / prev.AMT_APPLICATION.replace(0, np.nan)))
pagg = p.groupby('SK_ID_CURR').agg(
    PREV_N_APPS=('SK_ID_PREV', 'size'),
    PREV_REFUSED_SHARE=('refused', 'mean'),
    PREV_N_REFUSED=('refused', 'sum'),
    PREV_APPROVED_SHARE=('approved', 'mean'),
    PREV_CREDIT_TO_APP_MEAN=('got_less', 'mean'),
    PREV_DAYS_DECISION_MAX=('DAYS_DECISION', 'max'),
    PREV_CNT_PAYMENT_MEAN=('CNT_PAYMENT', 'mean'),
)
add(pagg); has_history('PREV_N_APPS')
del p, prev

### 10.3 `installments_payments` — actual repayment behaviour

In [ ]:
inst = pd.read_csv(DATA_DIR / 'installments_payments.csv', dtype='float32',
                   usecols=['SK_ID_CURR', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT'])
print(inst.shape)
days_late = (inst.DAYS_ENTRY_PAYMENT - inst.DAYS_INSTALMENT)
i = pd.DataFrame({'SK_ID_CURR': inst.SK_ID_CURR.astype('int64'),
                  'late': days_late > 0, 'days_late': days_late.clip(lower=0),
                  'underpaid': inst.AMT_PAYMENT < inst.AMT_INSTALMENT - 1})
iagg = i.groupby('SK_ID_CURR').agg(
    INST_N=('late', 'size'),
    INST_LATE_SHARE=('late', 'mean'),
    INST_DAYS_LATE_MEAN=('days_late', 'mean'),
    INST_DAYS_LATE_MAX=('days_late', 'max'),
    INST_UNDERPAID_SHARE=('underpaid', 'mean'),
)
add(iagg); has_history('INST_N')
del inst, i, days_late

### 10.4 `POS_CASH_balance` and `credit_card_balance` — monthly snapshots

In [ ]:
pos = pd.read_csv(DATA_DIR / 'POS_CASH_balance.csv', usecols=['SK_ID_CURR', 'SK_DPD', 'SK_DPD_DEF', 'MONTHS_BALANCE'])
print('POS', pos.shape)
posagg = pos.assign(dpd_pos=pos.SK_DPD_DEF > 0).groupby('SK_ID_CURR').agg(
    POS_N_MONTHS=('MONTHS_BALANCE', 'size'), POS_DPD_DEF_SHARE=('dpd_pos', 'mean'), POS_DPD_MAX=('SK_DPD', 'max'))
add(posagg); has_history('POS_N_MONTHS')
del pos

cc = pd.read_csv(DATA_DIR / 'credit_card_balance.csv',
                 usecols=['SK_ID_CURR', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'SK_DPD_DEF', 'AMT_DRAWINGS_CURRENT'])
print('credit_card', cc.shape)
cc['util'] = cc.AMT_BALANCE / cc.AMT_CREDIT_LIMIT_ACTUAL.replace(0, np.nan)
ccagg = cc.groupby('SK_ID_CURR').agg(
    CC_N_MONTHS=('util', 'size'), CC_UTIL_MEAN=('util', 'mean'), CC_UTIL_MAX=('util', 'max'),
    CC_DPD_DEF_SHARE=('SK_DPD_DEF', lambda s: (s > 0).mean()), CC_DRAWINGS_MEAN=('AMT_DRAWINGS_CURRENT', 'mean'))
add(ccagg); has_history('CC_N_MONTHS')
del cc

### 10.5 Which history signals matter?

In [ ]:
hcols = [c for c in hist.columns if c not in ('SK_ID_CURR', 'TARGET')]
hrank = pd.DataFrame({c: (*uni_auc(hist[c].astype(float), hist.TARGET), hist[c].notna().mean()) for c in hcols},
                     index=['uni_auc', 'direction', 'coverage']).T.sort_values('uni_auc', ascending=False)
print(hrank.to_string())

fig, ax = plt.subplots(figsize=(8, 6))
r = hrank.uni_auc.astype(float)[::-1]
ax.barh(r.index, r.values, color=C_BAR, height=0.6)
ax.set_xlim(0.5, max(0.65, r.max() + 0.01)); ax.grid(axis='y', visible=False)
ax.set_xlabel('univariate AUC (0.5 = no signal)'); ax.set_title('History-table aggregates - univariate AUC vs TARGET')
plt.tight_layout(); plt.show()

# default rate for clients with vs without each history type
cov = pd.DataFrame({
    name: {'with history': hist.TARGET[hist[c].notna()].mean(), 'without': hist.TARGET[hist[c].isna()].mean()}
    for name, c in [('bureau', 'BUR_N_CREDITS'), ('previous_app', 'PREV_N_APPS'), ('installments', 'INST_N'),
                    ('POS', 'POS_N_MONTHS'), ('credit_card', 'CC_N_MONTHS')]}).T
print('\nDefault rate with vs without each history type:'); print((cov * 100).round(2).to_string())

In [ ]:
for c in hrank.index[:4]:
    if hist[c].nunique() > 15:
        plot_numeric(c, df=hist)

### 10.6 Does history add signal on top of the application table?

In [ ]:
Xh = X.join(hist.set_index(train.index)[hcols])
Xh_tr, Xh_va = Xh.loc[X_tr.index], Xh.loc[X_va.index]
model_h = lgb.LGBMClassifier(**model.get_params())
model_h.fit(Xh_tr, y_tr, eval_set=[(Xh_va, y_va)], eval_metric='auc', callbacks=[lgb.early_stopping(100, verbose=False)])
va_auc_h = roc_auc_score(y_va, model_h.predict_proba(Xh_va)[:, 1])
print(f'Validation AUC - application only: {va_auc:.4f} | + {len(hcols)} history aggregates: {va_auc_h:.4f}')
imp_h = pd.Series(model_h.booster_.feature_importance('gain'), index=Xh.columns)
(imp_h / imp_h.sum()).sort_values(ascending=False).head(20).round(4).to_frame('gain_share')

## 11. Summary (computed)

In [ ]:
print(f'- {len(train):,} train / {len(test):,} test applications; default rate {rate:.2%} (heavily imbalanced -> use AUC, stratified CV).')
print(f'- {len(miss)} of {train.shape[1]} application columns have NaNs; {(miss > 0.5).sum()} are >50% missing.')
print(f'- DAYS_EMPLOYED placeholder 365243 in {anom.mean():.1%} of rows (pensioners); their default rate is lower.')
print(f'- Top features by gain: {", ".join(imp.index[:8])}.')
print(f'- Strongest single features (univariate AUC): ' +
      ', '.join(f'{c} {a:.3f}' for c, a in rank.uni_auc.sort_values(ascending=False).head(5).items()))
print(f'- Best history signals: ' + ', '.join(f'{c} {a:.3f}' for c, a in hrank.uni_auc.astype(float).head(5).items()))
print(f'- LightGBM val AUC: application only {va_auc:.4f} -> with history aggregates {va_auc_h:.4f}.')
print(f'- Max train/test KS on main features: {drift.ks_stat.max():.3f} ({drift.index[0]}).')

## 12. Handling the class imbalance (~8% defaults)

With ~25k positives this imbalance is mild in absolute terms. The metric (ROC-AUC) only measures how well the model ranks clients, so rebalancing the data rarely helps it. Plan:

1. **5-fold stratified CV** on application + history features (`Xh`). Every fold keeps the 8% rate, and out-of-fold (OOF) predictions are less noisy than the single split above.
2. **Compare class weighting** via `scale_pos_weight`: natural ratio (1), mild (3), fully balanced (neg/pos ≈ 11). Metrics: ROC-AUC, PR-AUC (focuses on the minority class) and Brier score / mean prediction (calibration).
3. **No SMOTE / oversampling.** Synthetic rows interpolated between defaulters don't suit tabular data with many NaNs and categoricals, they typically don't help gradient boosting, and they leak if applied before the CV split. Undersampling just discards data.
4. **Handle the imbalance at decision time.** Choose the approve/reject threshold from the OOF predictions with a business cost (a missed default costs much more than rejecting a good client), not 0.5.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve
from sklearn.calibration import calibration_curve

NEG_POS = (y == 0).sum() / (y == 1).sum()
CONFIGS = {'natural ratio (spw=1)': 1.0, 'mild weight (spw=3)': 3.0, f'balanced (spw={NEG_POS:.1f})': NEG_POS}

folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(Xh, y))
# metric='auc' only: otherwise early stopping also watches (unweighted) logloss, which degrades fast under
# class weights and stops the weighted models far too early - an unfair comparison.
base_params = {**model.get_params(), 'n_estimators': 5000, 'metric': 'auc'}
oof, cv_rows = {}, []
for name, spw in CONFIGS.items():
    t0 = time.time()
    p = np.zeros(len(Xh)); fold_auc, iters = [], []
    for tr, va in folds:
        m = lgb.LGBMClassifier(**{**base_params, 'scale_pos_weight': spw})
        m.fit(Xh.iloc[tr], y.iloc[tr], eval_set=[(Xh.iloc[va], y.iloc[va])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        p[va] = m.predict_proba(Xh.iloc[va])[:, 1]
        fold_auc.append(roc_auc_score(y.iloc[va], p[va])); iters.append(m.best_iteration_)
    oof[name] = p
    cv_rows.append({'config': name, 'oof_auc': roc_auc_score(y, p), 'fold_auc_std': np.std(fold_auc),
                    'pr_auc': average_precision_score(y, p), 'brier': brier_score_loss(y, p),
                    'mean_pred': p.mean(), 'avg_trees': int(np.mean(iters)), 'minutes': (time.time() - t0) / 60})
    print(f'{name:<24} OOF AUC {cv_rows[-1]["oof_auc"]:.4f}  ({cv_rows[-1]["minutes"]:.1f} min)')

cv = pd.DataFrame(cv_rows).set_index('config')
print(f'\nActual default rate {y.mean():.4f} - a calibrated model has mean_pred close to it. '
      f'PR-AUC of a random model = {y.mean():.3f}.')
cv.round(4)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for (name, p), c in zip(oof.items(), [C_BAR, '#1baf7a', C_DEFAULT]):
    prec, rec, _ = precision_recall_curve(y, p)
    a1.plot(rec, prec, color=c, label=f'{name}  AP {average_precision_score(y, p):.3f}')
    frac_pos, mean_p = calibration_curve(y, p, n_bins=20, strategy='quantile')
    a2.plot(mean_p, frac_pos, color=c, marker='o', markersize=3, label=name)
a1.axhline(y.mean(), color=INK2, linestyle='--', linewidth=1)
a1.text(1, y.mean(), 'random ', color=INK2, va='bottom', ha='right')
a1.set_xlabel('recall (share of defaulters caught)'); a1.set_ylabel('precision')
a1.set_title('Precision-recall (OOF)'); a1.legend()
a2.plot([0, 1], [0, 1], color=INK2, linestyle='--', linewidth=1)
a2.set_xlabel('predicted probability'); a2.set_ylabel('observed default rate')
a2.set_title('Calibration (OOF, 20 quantile bins)'); a2.legend()
plt.tight_layout(); plt.show()

### 12.1 Choosing the decision threshold

Defaults are rare, so 0.5 would reject almost no one. Instead, minimise the expected cost per application on the OOF predictions:

`cost = COST_FN · (approved defaulters) + COST_FP · (rejected good clients)`

`COST_FN / COST_FP` is a business input, roughly *loss given default × exposure* vs *margin lost on a good loan*; 5 is used as a placeholder. For calibrated probabilities the theoretical optimum is `COST_FP / (COST_FP + COST_FN)`, shown as a cross-check. The F1-optimal threshold is also shown for reference.

In [ ]:
COST_FN, COST_FP = 5.0, 1.0            # business input: cost of approving a defaulter vs rejecting a good client
CHOSEN = next(iter(CONFIGS))           # natural-ratio model: calibrated probabilities, so thresholds are interpretable
p, yv = oof[CHOSEN], y.values

rows = []
for t in np.round(np.arange(0.01, 0.61, 0.005), 3):
    rej = p >= t
    tp, fp, fn = (rej & (yv == 1)).sum(), (rej & (yv == 0)).sum(), (~rej & (yv == 1)).sum()
    rows.append({'threshold': t, 'reject_rate': rej.mean(), 'recall': tp / (tp + fn), 'precision': tp / max(tp + fp, 1),
                 'f1': 2 * tp / (2 * tp + fp + fn), 'cost_per_app': (COST_FN * fn + COST_FP * fp) / len(yv),
                 'default_rate_approved': fn / max((~rej).sum(), 1)})
th = pd.DataFrame(rows).set_index('threshold')
t_cost, t_f1, t_bayes = th.cost_per_app.idxmin(), th.f1.idxmax(), COST_FP / (COST_FP + COST_FN)

fig, ax = plt.subplots(figsize=(10, 3.8))
for col, c in [('recall', C_DEFAULT), ('precision', C_BAR), ('reject_rate', '#1baf7a')]:
    ax.plot(th.index, th[col], color=c, label=col)
ax2 = ax.twinx(); ax2.grid(False)
ax2.plot(th.index, th.cost_per_app, color=INK, linestyle=':', label=f'cost/app (FN:FP = {COST_FN:g}:{COST_FP:g})')
ax2.set_ylabel('expected cost per application')
for t, lab in [(t_cost, 'min cost'), (t_f1, 'max F1')]:
    ax.axvline(t, color=INK2, linestyle='--', linewidth=1)
    ax.text(t, 1.0, f' {lab} {t:.3f}', color=INK2, fontsize=8, va='top')
ax.set_xlabel('reject if predicted default probability >= threshold'); ax.set_ylim(0, 1.05)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc='center right'); ax.set_title(f'Threshold trade-offs - {CHOSEN}, OOF')
plt.tight_layout(); plt.show()

nearest = lambda t: th.index[np.abs(th.index - t).argmin()]
pick = th.loc[[nearest(t) for t in (t_cost, t_bayes, t_f1, 0.5)]]
pick.index = [f'min cost {t_cost:.3f}', f'Bayes 1/(1+{COST_FN / COST_FP:g}) {t_bayes:.3f}', f'max F1 {t_f1:.3f}', 'naive 0.5']
print(f'Base default rate {yv.mean():.2%}; approving everyone costs {COST_FN * yv.mean():.4f} per application.')
pick.round(4)

In [ ]:
best = cv.oof_auc.idxmax(); nat = cv.loc[CHOSEN]
gain = cv.oof_auc.max() - nat.oof_auc
print('Imbalance - conclusions (computed):')
print(f'- Best OOF AUC: {best} {cv.oof_auc.max():.4f}; natural ratio {nat.oof_auc:.4f} '
      f'(difference {gain:+.4f}, fold std ~{cv.fold_auc_std.mean():.4f}).')
print('  -> ' + ('class weighting gives a gain larger than the fold noise; consider keeping it (and calibrate if you need PDs).'
               if gain > 2 * cv.fold_auc_std.mean() else
               'class weighting does not beat fold noise: keep the natural ratio (simpler, calibrated probabilities).'))
print(f'- Calibration: actual rate {y.mean():.4f}; mean prediction ' +
      ', '.join(f'{k} {v:.4f}' for k, v in cv.mean_pred.items()) + '.')
r = th.loc[nearest(t_cost)]
print(f'- Decision rule (FN:FP = {COST_FN:g}:{COST_FP:g}): reject if p >= {t_cost:.3f} -> rejects {r.reject_rate:.1%} of applicants, '
      f'catches {r.recall:.1%} of defaulters, approved-book default rate {r.default_rate_approved:.2%} (vs {y.mean():.2%}).')